In [19]:
import pandas as pd
import os
from pathlib import Path

# Get all data files in the raw data folder
data_dir = Path("data/raw")
data_files = sorted(data_dir.glob("*.xls*"))

print(f"Found {len(data_files)} files in data/raw folder")
print("=" * 80)


Found 18 files in data/raw folder


In [20]:
# List all files
from pathlib import Path


print("\nData files found:")
for i, file in enumerate[Path](data_files, 1):
    print(f"{i}. {file.name}")
    
print("\n" + "=" * 80)



Data files found:
1. DAT20250713_1.xls
2. DAT20250713_2.xls
3. DAT20250714_1.xls
4. DAT20250714_2.xls
5. DAT20250715_1.xls
6. DAT20250715_2.xls
7. DAT20250716_1.xls
8. DAT20250716_2.xls
9. DAT20250717_1.xls
10. DAT20250717_2.xls
11. DAT20250718_1.xls
12. DAT20250718_2.xls
13. DAT20250719_1.xls
14. DAT20250719_2.xls
15. DAT20250720_1.xls
16. DAT20250720_2.xls
17. OSG - Raw Data_13au20juil2025.xlsx
18. ZOR001_BC_10Oct2025_Test_25Oct2025.xlsx



In [21]:
# Analyze DAT files (DAT20250713_1.xls format)
dat_files = [f for f in data_files if f.name.startswith('DAT')]
print(f"\nAnalyzing {len(dat_files)} DAT files...\n")

# Load first DAT file as sample with proper header parsing
sample_file = dat_files[0]
print(f"Sample file: {sample_file.name}")
print("=" * 80)

# Read header rows (row 0 = variable names, row 1 = units)
df_raw = pd.read_excel(sample_file, engine='xlrd', header=None, nrows=2)
var_names = df_raw.iloc[0].values
units = df_raw.iloc[1].values

# Create proper column names
column_names = []
for var, unit in zip(var_names, units):
    if pd.notna(var) and str(var).strip():
        if pd.notna(unit):
            column_names.append(f"{var} ({unit})")
        else:
            column_names.append(str(var))
    else:
        column_names.append(f"Column_{len(column_names)}")

# Read the actual data
df = pd.read_excel(sample_file, engine='xlrd', skiprows=2, names=column_names)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")



Analyzing 16 DAT files...

Sample file: DAT20250713_1.xls

Shape: 43,200 rows × 56 columns
Memory usage: 20.93 MB


In [22]:
# Display column names organized by category
print("\n" + "-" * 100)
print("COLUMN CATEGORIES:")
print("-" * 100)

col_categories = {
    'Date_Time': [col for col in column_names if 'Date_Time' in col],
    'M/E PORT (Main Engine Port)': [col for col in column_names if 'M/E PORT' in col],
    'M/E STBD (Main Engine Starboard)': [col for col in column_names if 'M/E STBD' in col],
    'Shipboard GPS': [col for col in column_names if 'GPS' in col],
    'Shipboard AIS': [col for col in column_names if 'AIS' in col],
    'LC-2 PORT': [col for col in column_names if 'LC-2 PORT' in col],
    'LC-2 STBD': [col for col in column_names if 'LC-2 STBD' in col],
    'Shaft PORT': [col for col in column_names if 'Shaft PORT' in col],
    'Shaft STBD': [col for col in column_names if 'Shaft STBD' in col]
}

for category, cols in col_categories.items():
    if cols:
        print(f"\n{category}: ({len(cols)} columns)")
        for col in cols:
            print(f"  - {col}")



----------------------------------------------------------------------------------------------------
COLUMN CATEGORIES:
----------------------------------------------------------------------------------------------------

Date_Time: (1 columns)
  - Date_Time

M/E PORT (Main Engine Port): (10 columns)
  - M/E PORT (Raw Supply Flowrate)
  - M/E PORT (Raw Supply Total Fuel)
  - M/E PORT (Raw Supply Temperature)
  - M/E PORT (Raw Return Flowrate)
  - M/E PORT (Raw Return Total Fuel)
  - M/E PORT (Raw Return Temperature)
  - M/E PORT (Flow Rate)
  - M/E PORT (Total Fuel)
  - M/E PORT (Temperature)
  - M/E PORT (Error)

M/E STBD (Main Engine Starboard): (10 columns)
  - M/E STBD (Raw Supply Flowrate)
  - M/E STBD (Raw Supply Total Fuel)
  - M/E STBD (Raw Supply Temperature)
  - M/E STBD (Raw Return Flowrate)
  - M/E STBD (Raw Return Total Fuel)
  - M/E STBD (Raw Return Temperature)
  - M/E STBD (Flow Rate)
  - M/E STBD (Total Fuel)
  - M/E STBD (Temperature)
  - M/E STBD (Error)

Shipboard 

In [23]:
# Display first few rows
print("\n" + "-" * 100)
print("FIRST 5 ROWS:")
print("-" * 100)
pd.set_option('display.max_columns', 8)
pd.set_option('display.width', 120)
print(df.head())



----------------------------------------------------------------------------------------------------
FIRST 5 ROWS:
----------------------------------------------------------------------------------------------------
             Date_Time  M/E PORT (Raw Supply Flowrate)  M/E PORT (Raw Supply Total Fuel)  \
0  2025-07-13 00:00:00                             0.0                       3112982.663   
1  2025-07-13 00:00:01                             0.0                       3112982.663   
2  2025-07-13 00:00:02                             0.0                       3112982.663   
3  2025-07-13 00:00:03                             0.0                       3112982.663   
4  2025-07-13 00:00:04                             0.0                       3112982.663   

   M/E PORT (Raw Supply Temperature)  ...  Shaft STBD (Torque)  Shaft STBD (Power)  Shaft STBD (Energy)  \
0                          43.600006  ...                  0.0                 0.0                  0.0   
1               

In [24]:
# Data types summary
print("\n" + "-" * 100)
print("DATA TYPES SUMMARY:")
print("-" * 100)
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"{dtype}: {count} columns")



----------------------------------------------------------------------------------------------------
DATA TYPES SUMMARY:
----------------------------------------------------------------------------------------------------
float64: 32 columns
int64: 23 columns
object: 1 columns


In [25]:
# Basic statistics
print("\n" + "-" * 100)
print("BASIC STATISTICS (Numeric columns):")
print("-" * 100)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
print(df[numeric_cols].describe().T.head(15))

# Missing values
print("\n" + "-" * 100)
print("MISSING VALUES:")
print("-" * 100)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
if len(missing_df) > 0:
    print(missing_df.head(10))
else:
    print("✓ No missing values found!")



----------------------------------------------------------------------------------------------------
BASIC STATISTICS (Numeric columns):
----------------------------------------------------------------------------------------------------
                                     count          mean         std           min           25%           50%  \
M/E PORT (Raw Supply Flowrate)     43200.0  1.056389e+02  207.498355  0.000000e+00  0.000000e+00  0.000000e+00   
M/E PORT (Raw Supply Total Fuel)   43200.0  3.113673e+06  583.489182  3.112983e+06  3.112983e+06  3.113860e+06   
M/E PORT (Raw Supply Temperature)  43200.0  3.896132e+01    5.497331  2.789999e+01  3.650000e+01  4.170001e+01   
M/E PORT (Raw Return Flowrate)     43200.0  9.311174e+01  180.194394  0.000000e+00  0.000000e+00  0.000000e+00   
M/E PORT (Raw Return Total Fuel)   43200.0  2.715802e+06  514.516889  2.715194e+06  2.715194e+06  2.715970e+06   
M/E PORT (Raw Return Temperature)  43200.0  4.414119e+01    4.183696  3.92000

In [26]:
# Analyze structure across all DAT files
print("\n" + "=" * 100)
print("CHECKING STRUCTURE ACROSS ALL DAT FILES")
print("=" * 100)

dat_info = []
for file in dat_files[:5]:  # Sample first 5 for performance
    try:
        df_temp = pd.read_excel(file, engine='xlrd', skiprows=2)
        dat_info.append({
            'File': file.name,
            'Rows': df_temp.shape[0],
            'Columns': df_temp.shape[1]
        })
    except Exception as e:
        dat_info.append({
            'File': file.name,
            'Rows': 0,
            'Columns': 0,
            'Error': str(e)[:50]
        })

summary_df = pd.DataFrame(dat_info)
print("\nSample of DAT Files (first 5):")
print(summary_df.to_string(index=False))
if len(dat_files) > 5:
    print(f"\n... and {len(dat_files) - 5} more DAT files")



CHECKING STRUCTURE ACROSS ALL DAT FILES

Sample of DAT Files (first 5):
             File  Rows  Columns
DAT20250713_1.xls 43200       56
DAT20250713_2.xls 43200       56
DAT20250714_1.xls 43200       56
DAT20250714_2.xls 43200       56
DAT20250715_1.xls 43006       56

... and 11 more DAT files


In [27]:
# Analyze other files
print("\n" + "=" * 100)
print("ANALYZING OTHER FILES")
print("=" * 100)
other_files = [f for f in data_files if not f.name.startswith('DAT')]

for file in other_files:
    print(f"\n{'='*80}")
    print(f"File: {file.name}")
    print('='*80)
    try:
        df_other = pd.read_excel(file, engine='openpyxl')
        print(f"Shape: {df_other.shape[0]:,} rows × {df_other.shape[1]} columns")
        print(f"\nColumn names ({len(df_other.columns)}):")
        for i, col in enumerate(df_other.columns[:25], 1):
            print(f"{i:2d}. {col}")
        if len(df_other.columns) > 25:
            print(f"   ... and {len(df_other.columns) - 25} more columns")
        print(f"\nFirst 3 rows:")
        print(df_other.head(3))
    except Exception as e:
        print(f"Error reading file: {e}")



ANALYZING OTHER FILES

File: OSG - Raw Data_13au20juil2025.xlsx
Shape: 11,506 rows × 25 columns

Column names (25):
 1. Time_Epoch
 2. Date
 3. Duration 1 min
 4. Ship_Name
 5. Operation_Mode
 6. Voyage_Zone
 7. Latitude
 8. Longitude
 9. SOG (kn)
10. Fuel_Flow_PORT (L/h)
11. Fuel_Flow_STBD (L/h)
12. Flow_Temp_PORT (degC)
13. Flow_Temp_STBD (degC)
14. Fuel_Total_PORT (L)
15. Fuel_Total_STBD (L)
16. Torque_PORT (kN*m)
17. Torque_STBD (kN*m)
18. Power_PORT (kW)
19. Energy_Port (kWh)
20. Power_STBD (kW)
21. Energy_STBD (kWh)
22. RPM_PORT
23. RPM_STBD
24. LC2 PORT Oxygen PORT
25. LC2 STBD Oxygen STBD

First 3 rows:
   Time_Epoch                Date  Duration 1 min Ship_Name  ... RPM_PORT RPM_STBD  LC2 PORT Oxygen PORT  \
0  1752379200 2025-07-13 00:00:00               1  OSGenois  ...      0.0      0.0                     0   
1  1752379260 2025-07-13 00:01:00               1  OSGenois  ...      0.0      0.0                     0   
2  1752379320 2025-07-13 00:02:00               1  OSGen

In [28]:
# Comprehensive Summary
print("\n" + "=" * 100)
print("COMPREHENSIVE DATA SUMMARY")
print("=" * 100)

print(f"\nTotal files found: {len(data_files)}")
print(f"  - DAT files: {len(dat_files)}")
print(f"  - Other files: {len(other_files)}")

print(f"\nSample DAT file analysis ({sample_file.name}):")
print(f"  - Total rows: {df.shape[0]:,}")
print(f"  - Total columns: {df.shape[1]}")
print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  - Data types: {type_counts.to_dict()}")

print("\nKey findings:")
print("  - DAT files contain time-series data with 1-second intervals")
print("  - Data includes: Main Engine (PORT/STBD), GPS, AIS, LC-2 sensors, Shaft measurements")
print("  - Each DAT file covers approximately 12 hours of data (43,200 seconds)")

print("\n✓ Analysis complete!")



COMPREHENSIVE DATA SUMMARY

Total files found: 18
  - DAT files: 16
  - Other files: 2

Sample DAT file analysis (DAT20250713_1.xls):
  - Total rows: 43,200
  - Total columns: 56
  - Memory usage: 20.93 MB
  - Data types: {dtype('float64'): 32, dtype('int64'): 23, dtype('O'): 1}

Key findings:
  - DAT files contain time-series data with 1-second intervals
  - Data includes: Main Engine (PORT/STBD), GPS, AIS, LC-2 sensors, Shaft measurements
  - Each DAT file covers approximately 12 hours of data (43,200 seconds)

✓ Analysis complete!
